# Train the Transformer (DonderFormer)

An encoder-decoder Transformer that writes a chart as event tokens (see `src/donderformer/models/donderformer.py` and `data/tokens.py`):

- **Encoder** reads a window of `past + target + lookahead` mel frames (default 512 + 256 + 64 = 832 frames, 19.1 s).
- **Decoder** sees `BOS <course> <level> <past events> SEP` and generates the target region's events as `TIME_k NOTE` pairs, where `TIME_k` is frame `k` of the window (23 ms grid).
- **Generation** slides the window by `target` frames; each window's past region is primed with the notes generated so far.

The output is the shared note format (one class id per 23 ms frame). This notebook only trains: test generation and scoring happen in `05_evaluation`, with the same scorer as the LSTM.

Set `TOY = True` for a quick end-to-end check on the toy dataset (tiny model, a few epochs, CPU is fine).

In [ ]:
import math
import time
from dataclasses import asdict

import numpy as np
import torch
from torch.utils.data import DataLoader

from donderformer.data.chart_dataset import Augment, ChartWindowDataset, WindowConfig, collate
from donderformer.data.feature_extractor import FeatureExtractor
from donderformer.models.donderformer import DonderFormer, DonderFormerConfig
from donderformer.paths import DataPaths, outputs_dir
from donderformer.runs import load_run, new_run_dir, save_run
from donderformer.training import evaluate_tokens, train_epoch_tokens

TOY = False                                   # True: toy dataset, tiny model, short run

if TOY:
    paths = DataPaths.toy()
    SPLIT = "toy_all"                         # every course of the 12 toy songs
    PRESET = "tiny"
    EPOCHS, BATCH_SIZE, WARMUP_STEPS = 5, 16, 20       # a pipeline check, not a real run
    EVAL_DECODE_EVERY = 5
else:
    paths = DataPaths()                       # DONDER_DATA env var, else <repo>/data
    SPLIT = "oni_v2"                          # all_v1 to pretrain on every course
    PRESET = "base"                           # base (~11.5M params), small (~4.3M), tiny (~1M)
    EPOCHS, BATCH_SIZE, WARMUP_STEPS = 200, 64, 1000
    EVAL_DECODE_EVERY = 5

extractor = FeatureExtractor(frame_ms=23)     # cache made by 02_extract_features
WINDOW = WindowConfig(past_frames=512, target_frames=256, lookahead_frames=64, max_tokens=640)
MODEL_CFG = DonderFormerConfig.preset(PRESET)
AUGMENT = Augment()                           # SpecAugment, level dropout, prefix corruption

LR = 3e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.02
PATIENCE = 15                                 # epochs without a validation-loss gain before stopping
SEED = 0
INIT_FROM = None                              # run folder to start from, e.g. the all_v1 pretrain run
SAMPLING = dict(temperature=1.0, top_p=0.9)   # saved for 06's test generation; validation decoding is greedy

MODEL_NAME = "donderformer"
ALIAS = PRESET + ("_toy" if TOY else "")
RUN_DIR = new_run_dir(MODEL_NAME, SPLIT, extractor.id, alias=ALIAS)
print("Run folder:", RUN_DIR)
HPARAMS = dict(preset=PRESET, epochs=EPOCHS, batch_size=BATCH_SIZE, warmup_steps=WARMUP_STEPS, lr=LR,
               min_lr=MIN_LR, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING, patience=PATIENCE,
               sampling=SAMPLING, augment=AUGMENT.__dict__, seed=SEED,
               init_from=str(INIT_FROM) if INIT_FROM else None)
torch.manual_seed(SEED)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
# bf16 autocast on CUDA (A100/H100/H200); full precision elsewhere.
AMP_DTYPE = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else None
print("Device:", device, "| autocast:", AMP_DTYPE)
if device.type == "cuda":
    print(torch.cuda.get_device_name(device))

## Data

Training windows start at random frames and are redrawn every epoch (`resample`); validation and test windows tile each chart with hop `target_frames`. All mel of a part is held in memory (float16, about 1.4 GB for the full Oni train part).

In [ ]:
train_dataset = ChartWindowDataset(paths, SPLIT, extractor, "train", WINDOW, augment=AUGMENT, seed=SEED)
val_dataset = ChartWindowDataset(paths, SPLIT, extractor, "validation", WINDOW)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate)
for name, ds in (("train", train_dataset), ("validation", val_dataset)):
    print(f"{name}: {len(ds.chart_ids)} charts, {len(ds)} windows")

# One training example, decoded
mel, tokens, sep = val_dataset[min(2, len(val_dataset) - 1)]
print("mel", tuple(mel.shape), "| tokens", len(tokens), "| SEP at", sep)
print(val_dataset.vocab.decode(tokens[:40]), "...")

## Model

In [ ]:
model = DonderFormer(MODEL_CFG, WINDOW).to(device)
if INIT_FROM:
    load_run(INIT_FROM, model, map_location=device)   # same PRESET and WINDOW as that run
    print("Initialized from", INIT_FROM)
print(f"{PRESET}: {model.n_parameters() / 1e6:.2f}M parameters, vocab {model.vocab.size}")

# Sanity check: shapes, and the loss starts near log(vocab)
mel, tokens, sep = collate([train_dataset[i] for i in range(4)])
with torch.no_grad():
    logits = model(mel.to(device), tokens.to(device))
print("logits", tuple(logits.shape), "| initial loss",
      round(evaluate_tokens(model, [(mel, tokens, sep)], device), 3), "vs log(vocab)", round(math.log(model.vocab.size), 3))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), weight_decay=WEIGHT_DECAY)
TOTAL_STEPS = EPOCHS * len(train_loader)

def lr_at(step):
    """Linear warmup, then cosine down to MIN_LR."""
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS))
    return (MIN_LR + (LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * progress))) / LR

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)

## Validation decoding

Onset and onset+type F1 come from `donderformer.evaluation`, the same functions `05_evaluation` uses for the test set.

In [ ]:
from donderformer.evaluation import donderformer_chart, joint_onset_type_metrics, onset_metrics


def decode_f1(dataset, n_charts=None):
    """Mean onset and onset+type F1 of greedily generated charts."""
    scores = []
    for chart_id in dataset.chart_ids[:n_charts]:
        human, generated = donderformer_chart(model, dataset, chart_id, seed=0, sampling=dict(temperature=0.0))
        scores.append((onset_metrics(human, generated)[2], joint_onset_type_metrics(human, generated)[2]))
    return tuple(float(x) for x in np.mean(scores, axis=0))

## Train

Validation loss is on target-region tokens only. Every `EVAL_DECODE_EVERY` epochs a few validation charts are generated greedily and scored. Each new best validation loss is saved to the run folder right away (`model.pt`, `config.json`), so a run cut off by the walltime keeps its best model. Training stops after `PATIENCE` epochs without improvement.

In [ ]:
import copy

RUN_CONFIG = {"model": MODEL_NAME, "alias": ALIAS,
              "data": {"split": SPLIT, "features": extractor.id, "extractor": asdict(extractor)},
              "window": WINDOW.as_dict(), "model_config": MODEL_CFG.as_dict(), "hparams": HPARAMS,
              "n_parameters": model.n_parameters(), "device": str(device)}
history = {"train": [], "val": [], "val_decode": []}
best = {"val": float("inf"), "epoch": -1, "state": None}
start_time = time.time()

for epoch in range(EPOCHS):
    train_dataset.resample(epoch)
    train_loss = train_epoch_tokens(model, train_loader, optimizer, device, scheduler,
                                    label_smoothing=LABEL_SMOOTHING, amp_dtype=AMP_DTYPE)
    val_loss = evaluate_tokens(model, val_loader, device, amp_dtype=AMP_DTYPE)
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    line = (f"Epoch {epoch + 1:03d} | train {train_loss:.4f} | val {val_loss:.4f} | "
            f"lr {scheduler.get_last_lr()[0]:.2e} | {(time.time() - start_time) / 60:.1f} min")

    if (epoch + 1) % EVAL_DECODE_EVERY == 0:
        onset_f1, joint_f1 = decode_f1(val_dataset, n_charts=8)
        history["val_decode"].append({"epoch": epoch + 1, "onset_f1": onset_f1, "joint_f1": joint_f1})
        line += f" | val greedy onset F1 {onset_f1:.3f}, joint {joint_f1:.3f}"

    if val_loss < best["val"]:
        best = {"val": val_loss, "epoch": epoch + 1, "state": copy.deepcopy(model.state_dict())}
        save_run(RUN_DIR, model, config={**RUN_CONFIG, "best_epoch": best["epoch"]}, losses=history)
        line += " | saved"
    print(line, flush=True)
    if epoch + 1 - best["epoch"] >= PATIENCE:
        print(f"No validation gain for {PATIENCE} epochs; stopping")
        break

model.load_state_dict(best["state"])
print(f"Best epoch {best['epoch']}, val loss {best['val']:.4f}")

In [ ]:
checkpoint = save_run(RUN_DIR, model, config={**RUN_CONFIG, "best_epoch": best["epoch"]}, losses=history)
print("Saved", checkpoint)

## Next: evaluate

Set `RUN_DIR` in `05_evaluation` to the folder printed above. It generates every test chart with the saved `sampling`, scores them with the shared scorer, writes `eval/ar_test/` (charts, `analysis/results.json`), and `06_compare_models` can then pair this run with the LSTM. Charts can be exported to TJA from `07_exports`.